## 1D and 2D plots along P2-P1 or in a rectangular region perpendicular to P2-P1

In [1]:
from pathlib import Path
import numpy as np
from scipy.constants import physical_constants
from scipy.interpolate import interp1d

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

from useful_notebooks_cube import (
    cube_plane_average_profile,
    cube_perpendicular_plane_map,
    bohr_to_ang,
    plot_line_profile,
    plot_plane_map,
    parse_point,
    float_or_none,
    collect_cube_files,
    read_cube_full_cached,
)
from useful_notebooks_cube import parse_point, float_or_none, collect_cube_files

In [3]:
def _update_visibility(*args):
    if plot_type_dropdown.value == "1d":
        dl_text.layout.display = ""
        P0_text.layout.display = "none"
        sigma_text.layout.display = "none"
        dn_text.layout.display = "none"

        row5_1d.layout.display = ""
        row5_2d.layout.display = "none"
        row6_2d.layout.display = "none"
    else:
        dl_text.layout.display = "none"
        P0_text.layout.display = ""
        sigma_text.layout.display = ""
        dn_text.layout.display = ""

        row5_1d.layout.display = "none"
        row5_2d.layout.display = ""
        row6_2d.layout.display = ""

def _read_common_inputs():
    filename = cube_file_dropdown.value
    if not filename:
        raise ValueError("No cube file is available in the selected directory.")

    field_type = field_type_dropdown.value
    plot_type = plot_type_dropdown.value
    order = int(interp_order.value)

    P1 = parse_point(P1_text.value)
    P2 = parse_point(P2_text.value)

    if np.allclose(P1, P2):
        raise ValueError("P1 and P2 must be different.")

    L_val = float_or_none(L_text.value)
    W_val = float_or_none(W_text.value)

    header_lines, _, cube_values, grid_shape = read_cube_full_cached(filename,verbose=True)

    return {
        "filename": filename,
        "field_type": field_type,
        "plot_type": plot_type,
        "order": order,
        "P1": P1,
        "P2": P2,
        "L_val": L_val,
        "W_val": W_val,
        "header_lines": header_lines,
        "cube_values": cube_values,
        "grid_shape": grid_shape,
    }

def _populate_LW_if_needed(result, common):
    if common["L_val"] is None:
        L_text.value = f"{result['L_default_ang']:.6g}"
    if common["W_val"] is None:
        W_text.value = f"{result['W_default_ang']:.6g}"

def _print_common_summary(result, common):
    print(f"Grid shape: {common['grid_shape']}")
    print(f"L used = {result['L_ang']:.6g} Å")
    print(f"W used = {result['W_ang']:.6g} Å")

def _run_1d_plot(common):
    dl_val = float(dl_text.value)

    result = cube_plane_average_profile(
        header_lines=common["header_lines"],
        cube_values=common["cube_values"],
        bohr_to_ang=bohr_to_ang,
        P1=common["P1"],
        P2=common["P2"],
        field_type=common["field_type"],
        dl=dl_val,
        L=common["L_val"],
        W=common["W_val"],
        order=common["order"],
    )

    _populate_LW_if_needed(result, common)

    plot_line_profile(
        result,
        title=Path(common["filename"]).name,
    )

    _print_common_summary(result, common)
    print(f"Rectangle area = {result['rectangle_area_ang2']:.6g} Å²")
    print(f"dl used = {result['dl_ang']:.6g} Å")
    

def _run_2d_plot(common):
    axis_length = np.linalg.norm(common["P2"] - common["P1"])

    P0_val = float_or_none(P0_text.value)
    if P0_val is None:
        P0_val = 0.5 * axis_length
        P0_text.value = f"{P0_val:.6g}"

    sigma_val = float(sigma_text.value)
    dn_val = float(dn_text.value)

    result = cube_perpendicular_plane_map(
        header_lines=common["header_lines"],
        cube_values=common["cube_values"],
        bohr_to_ang=bohr_to_ang,
        P1=common["P1"],
        P2=common["P2"],
        position=P0_val,
        field_type=common["field_type"],
        L=common["L_val"],
        W=common["W_val"],
        sigma=sigma_val,
        dn=dn_val,
        order=common["order"],
    )

    _populate_LW_if_needed(result, common)

    plot_plane_map(
        result,
        title=f"{Path(common['filename']).name}   |   P0 = {result['position_ang']:.6g} Å",
    )

    _print_common_summary(result, common)
    print(f"P0 used = {result['position_ang']:.6g} Å")
    print(f"sigma used = {result['sigma_ang']:.6g} Å")
    print(f"dn used = {result['dn_ang']:.6g} Å")

def _on_plot_clicked(_):
    with output:
        clear_output()

        try:
            common = _read_common_inputs()

            if common["plot_type"] == "1d":
                _run_1d_plot(common)
            else:
                _run_2d_plot(common)

        except Exception as exc:
            print(f"Error: {exc}")

In [4]:
# ----------------------------
# read cube file
# ----------------------------
downloads_dir = Path.cwd() / "Downloads"
cube_files = collect_cube_files(downloads_dir)

# ----------------------------
# widgets
# ----------------------------
if cube_files:
    cube_file_dropdown = widgets.Dropdown(
        options=[(p.name, str(p)) for p in cube_files],
        description="Cube file:",
        layout=widgets.Layout(width="700px"),
        style={"description_width": "120px"},
    )
else:
    cube_file_dropdown = widgets.Dropdown(
        options=[(f"No .cube/.cub files found in {downloads_dir}", "")],
        description="Cube file:",
        layout=widgets.Layout(width="700px"),
        style={"description_width": "120px"},
        disabled=True,
    )
    
#cube_file_dropdown.observe(_on_cube_file_change, names="value")

field_type_dropdown = widgets.Dropdown(
    options=[
        ("charge", "charge"),
        ("hartree", "hartree"),
        ("rydberg", "rydberg"),
    ],
    value="charge",
    description="Field type:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "120px"},
)

plot_type_dropdown = widgets.Dropdown(
    options=[
        ("1D line plot", "1d"),
        ("2D plot", "2d"),
    ],
    value="1d",
    description="Plot type:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "120px"},
)

P1_text = widgets.Text(
    value="0 0 0",
    description="P1 (Å):",
    placeholder="e.g. 0 0 0",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "120px"},
)

P2_text = widgets.Text(
    value="0 0 10",
    description="P2 (Å):",
    placeholder="e.g. 0 0 10",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "120px"},
)

L_text = widgets.Text(
    value="",
    description="L (Å):",
    placeholder="empty = use default",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

W_text = widgets.Text(
    value="",
    description="W (Å):",
    placeholder="empty = use default",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

dl_text = widgets.Text(
    value="0.1",
    description="dl (Å):",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

P0_text = widgets.Text(
    value="",
    description="P0 (Å):",
    placeholder="empty = midpoint between P1 and P2",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "120px"},
)

interp_order = widgets.BoundedIntText(
    value=1,
    min=0,
    max=5,
    step=1,
    description="Interp. order:",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

sigma_text = widgets.Text(
    value="0.1",
    description="sigma (Å):",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

dn_text = widgets.Text(
    value="3",
    description="dn (Å):",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

plot_button = widgets.Button(
    description="Plot",
    button_style="primary",
    layout=widgets.Layout(width="120px"),
)

output = widgets.Output()

row1 = widgets.HBox([cube_file_dropdown])
row2 = widgets.HBox([field_type_dropdown, plot_type_dropdown])
row3 = widgets.HBox([P1_text, P2_text])
row4 = widgets.HBox([L_text, W_text])
row5_1d = widgets.HBox([dl_text, interp_order])
row5_2d = widgets.HBox([P0_text, interp_order])
row6_2d = widgets.HBox([sigma_text, dn_text])

plot_type_dropdown.observe(_update_visibility, names="value")
_update_visibility()
        
plot_button.on_click(_on_plot_clicked)

ui = widgets.VBox([
    row1,
    row2,
    row3,
    row4,
    row5_1d,
    row5_2d,
    row6_2d,
    plot_button,
    output,
])

display(ui)